# Advanced Louvain Method for Community Detection

This notebook provides comprehensive Louvain method implementation with advanced network visualizations for methodology analysis.

## Features:
- Network layout graphs with force-directed layouts
- Modularity evolution tracking over iterations
- Community size distribution analysis
- Hierarchical community structure visualization
- Interactive network graphs with zoom, pan, and hover
- Multi-resolution analysis

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
import cv2
from sklearn.datasets import make_blobs
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import pandas as pd
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# Try to import community detection libraries
try:
    import community as community_louvain
    print("python-louvain imported successfully")
except ImportError:
    print("python-louvain not available, installing...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-louvain"])
    import community as community_louvain

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries imported successfully!")

## Enhanced Louvain Implementation with Tracking

In [ ]:
class EnhancedLouvain:
    """Enhanced Louvain method with comprehensive tracking and visualization."""
    
    def __init__(self, resolution=1.0, random_state=42):
        self.resolution = resolution
        self.random_state = random_state
        
        # Tracking variables
        self.modularity_history = []
        self.community_history = []
        self.level_graphs = []
        self.hierarchy_levels = []
        self.iteration_times = []
        
    def fit(self, graph):
        """Fit Louvain algorithm with detailed tracking."""
        np.random.seed(self.random_state)
        
        current_graph = graph.copy()
        level = 0
        
        # Store original graph
        self.original_graph_ = graph.copy()
        
        while True:
            start_time = time.time()
            
            # Apply Louvain algorithm at current level
            partition = community_louvain.best_partition(current_graph, 
                                                       resolution=self.resolution,
                                                       random_state=self.random_state)
            
            # Calculate modularity
            modularity = community_louvain.modularity(partition, current_graph)
            
            # Store results
            self.modularity_history.append(modularity)
            self.community_history.append(partition.copy())
            self.level_graphs.append(current_graph.copy())
            self.iteration_times.append(time.time() - start_time)
            
            # Create next level graph (induced graph)
            next_graph = community_louvain.induced_graph(partition, current_graph)
            
            # Check if we should continue (improvement in modularity)
            if len(self.modularity_history) > 1:
                improvement = (self.modularity_history[-1] - self.modularity_history[-2])
                if improvement < 1e-6:  # Very small improvement
                    break
            
            current_graph = next_graph
            level += 1
            
            if level > 10:  # Prevent infinite loops
                break
        
        # Final results
        self.best_partition_ = self.community_history[-1]
        self.modularity_ = self.modularity_history[-1]
        self.n_levels_ = len(self.modularity_history)
        
        # Create labels array
        self.labels_ = np.array([self.best_partition_[node] for node in sorted(graph.nodes())])
        
        return self
    
    def get_hierarchy_at_level(self, level):
        """Get community assignment at specific hierarchy level."""
        if level < len(self.community_history):
            return self.community_history[level]
        else:
            return self.community_history[-1]

print("Enhanced Louvain class defined!")

## Network Construction Utilities

In [ ]:
def create_network_from_data(X, method='knn', k=10, threshold=0.5):
    """Create network graph from data points."""
    n_samples = len(X)
    
    if method == 'knn':
        # K-nearest neighbors graph
        knn_graph = kneighbors_graph(X, n_neighbors=k, mode='connectivity')
        # Make symmetric
        adjacency = (knn_graph + knn_graph.T) / 2
        
    elif method == 'threshold':
        # Threshold-based graph (Euclidean distance)
        from sklearn.metrics import pairwise_distances
        distances = pairwise_distances(X)
        adjacency = (distances < threshold).astype(float)
        np.fill_diagonal(adjacency, 0)  # Remove self-loops
        
    elif method == 'epsilon':
        # Epsilon-neighborhood graph
        from sklearn.neighbors import radius_neighbors_graph
        adjacency = radius_neighbors_graph(X, radius=threshold, mode='connectivity')
        
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Convert to NetworkX graph
    G = nx.from_scipy_sparse_array(adjacency)
    
    # Add node positions as attributes
    pos_dict = {i: (X[i, 0], X[i, 1]) for i in range(len(X))}
    nx.set_node_attributes(G, pos_dict, 'pos')
    
    return G

def create_sample_networks():
    """Create sample networks for demonstration."""
    networks = []
    
    # 1. Karate Club (classic network)
    karate = nx.karate_club_graph()
    networks.append((karate, "Karate Club"))
    
    # 2. Random geometric graph
    geometric = nx.random_geometric_graph(100, 0.2, seed=42)
    networks.append((geometric, "Random Geometric"))
    
    # 3. Barabási-Albert preferential attachment
    barabasi = nx.barabasi_albert_graph(100, 3, seed=42)
    networks.append((barabasi, "Barabási-Albert"))
    
    # 4. Erdős-Rényi random graph
    erdos = nx.erdos_renyi_graph(100, 0.05, seed=42)
    networks.append((erdos, "Erdős-Rényi"))
    
    return networks

print("Network construction utilities defined!")

## Modularity Evolution Visualization

In [ ]:
def plot_modularity_evolution(louvain_model, title="Modularity Evolution Analysis"):
    """Plot comprehensive modularity evolution analysis."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    levels = range(len(louvain_model.modularity_history))
    
    # 1. Modularity over hierarchy levels
    axes[0, 0].plot(levels, louvain_model.modularity_history, 
                    marker='o', linewidth=3, markersize=8, color='blue')
    axes[0, 0].set_title('Modularity Evolution')
    axes[0, 0].set_xlabel('Hierarchy Level')
    axes[0, 0].set_ylabel('Modularity')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Highlight best modularity
    best_level = np.argmax(louvain_model.modularity_history)
    best_modularity = louvain_model.modularity_history[best_level]
    axes[0, 0].axvline(best_level, color='red', linestyle='--', alpha=0.7,
                       label=f'Best: Level {best_level} (Q={best_modularity:.3f})')
    axes[0, 0].legend()
    
    # 2. Modularity improvement per level
    if len(louvain_model.modularity_history) > 1:
        improvements = np.diff(louvain_model.modularity_history)
        axes[0, 1].bar(range(1, len(levels)), improvements, 
                       color=plt.cm.viridis(np.linspace(0, 1, len(improvements))))
        axes[0, 1].set_title('Modularity Improvement per Level')
        axes[0, 1].set_xlabel('Hierarchy Level')
        axes[0, 1].set_ylabel('Modularity Gain')
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Number of communities per level
    n_communities = [len(set(partition.values())) for partition in louvain_model.community_history]
    axes[1, 0].plot(levels, n_communities, marker='s', linewidth=3, markersize=8, color='orange')
    axes[1, 0].set_title('Number of Communities per Level')
    axes[1, 0].set_xlabel('Hierarchy Level')
    axes[1, 0].set_ylabel('Number of Communities')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Processing time per level
    if louvain_model.iteration_times:
        axes[1, 1].bar(levels, louvain_model.iteration_times, 
                       color=plt.cm.plasma(np.linspace(0, 1, len(levels))))
        axes[1, 1].set_title('Processing Time per Level')
        axes[1, 1].set_xlabel('Hierarchy Level')
        axes[1, 1].set_ylabel('Time (seconds)')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return best_level, best_modularity

print("Modularity evolution visualization function defined!")

## Community Structure Analysis

In [ ]:
def plot_community_analysis(graph, partition, title="Community Structure Analysis"):
    """Comprehensive community structure analysis."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Get community information
    communities = {}
    for node, comm in partition.items():
        if comm not in communities:
            communities[comm] = []
        communities[comm].append(node)
    
    community_sizes = [len(members) for members in communities.values()]
    n_communities = len(communities)
    
    # 1. Community size distribution
    axes[0, 0].hist(community_sizes, bins=min(20, n_communities), 
                    alpha=0.7, edgecolor='black', color='skyblue')
    axes[0, 0].set_title('Community Size Distribution')
    axes[0, 0].set_xlabel('Community Size')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].axvline(np.mean(community_sizes), color='red', linestyle='--',
                       label=f'Mean: {np.mean(community_sizes):.1f}')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Community size vs community ID
    comm_ids = sorted(communities.keys())
    sizes_by_id = [len(communities[cid]) for cid in comm_ids]
    
    axes[0, 1].bar(comm_ids, sizes_by_id, 
                   color=plt.cm.Set3(np.linspace(0, 1, len(comm_ids))))
    axes[0, 1].set_title('Community Sizes by ID')
    axes[0, 1].set_xlabel('Community ID')
    axes[0, 1].set_ylabel('Size')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Intra vs inter-community edges
    intra_edges = 0
    inter_edges = 0
    
    for edge in graph.edges():
        node1, node2 = edge
        if partition[node1] == partition[node2]:
            intra_edges += 1
        else:
            inter_edges += 1
    
    edge_types = ['Intra-community', 'Inter-community']
    edge_counts = [intra_edges, inter_edges]
    colors = ['lightcoral', 'lightsteelblue']
    
    wedges, texts, autotexts = axes[1, 0].pie(edge_counts, labels=edge_types, 
                                             colors=colors, autopct='%1.1f%%',
                                             startangle=90)
    axes[1, 0].set_title('Edge Distribution')
    
    # 4. Community connectivity matrix
    # Create inter-community adjacency matrix
    comm_adj = np.zeros((n_communities, n_communities))
    
    for edge in graph.edges():
        node1, node2 = edge
        comm1, comm2 = partition[node1], partition[node2]
        comm_adj[comm1, comm2] += 1
        if comm1 != comm2:
            comm_adj[comm2, comm1] += 1
    
    if n_communities <= 20:  # Only show for reasonable number of communities
        im = axes[1, 1].imshow(comm_adj, cmap='Blues')
        axes[1, 1].set_title('Inter-Community Connectivity')
        axes[1, 1].set_xlabel('Community ID')
        axes[1, 1].set_ylabel('Community ID')
        plt.colorbar(im, ax=axes[1, 1])
    else:
        axes[1, 1].text(0.5, 0.5, f'Too many communities\nto display matrix\n({n_communities} communities)',
                        ha='center', va='center', transform=axes[1, 1].transAxes,
                        fontsize=12)
        axes[1, 1].set_title('Inter-Community Connectivity')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary statistics
    print(f"\nCommunity Structure Summary:")
    print(f"  Number of communities: {n_communities}")
    print(f"  Average community size: {np.mean(community_sizes):.2f}")
    print(f"  Largest community: {max(community_sizes)} nodes")
    print(f"  Smallest community: {min(community_sizes)} nodes")
    print(f"  Intra-community edges: {intra_edges} ({intra_edges/(intra_edges+inter_edges)*100:.1f}%)")
    print(f"  Inter-community edges: {inter_edges} ({inter_edges/(intra_edges+inter_edges)*100:.1f}%)")
    
    return communities, intra_edges, inter_edges

print("Community analysis function defined!")

## Network Layout Visualization

In [ ]:
def plot_network_layouts(graph, partition, title="Network Layout Visualization"):
    """Create multiple network layout visualizations."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Get unique communities and assign colors
    communities = list(set(partition.values()))
    colors = plt.cm.Set3(np.linspace(0, 1, len(communities)))
    node_colors = [colors[partition[node]] for node in graph.nodes()]
    
    # Layout algorithms
    layouts = [
        (nx.spring_layout(graph, seed=42), "Spring Layout (Force-directed)"),
        (nx.kamada_kawai_layout(graph), "Kamada-Kawai Layout"),
        (nx.circular_layout(graph), "Circular Layout"),
        (nx.spectral_layout(graph), "Spectral Layout")
    ]
    
    for idx, (pos, layout_name) in enumerate(layouts):
        row, col = idx // 2, idx % 2
        
        # Draw network
        nx.draw_networkx_nodes(graph, pos, node_color=node_colors, 
                              node_size=50, alpha=0.8, ax=axes[row, col])
        nx.draw_networkx_edges(graph, pos, alpha=0.3, width=0.5, ax=axes[row, col])
        
        axes[row, col].set_title(layout_name)
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

def create_interactive_network_visualization(graph, partition, layout='spring', 
                                           title="Interactive Network Visualization"):
    """Create interactive network visualization with Plotly."""
    # Choose layout
    if layout == 'spring':
        pos = nx.spring_layout(graph, seed=42)
    elif layout == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(graph)
    elif layout == 'circular':
        pos = nx.circular_layout(graph)
    else:
        pos = nx.spring_layout(graph, seed=42)
    
    # Create edge traces
    edge_x = []
    edge_y = []
    
    for edge in graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
    
    edge_trace = go.Scatter(x=edge_x, y=edge_y,
                           line=dict(width=0.5, color='#888'),
                           hoverinfo='none',
                           mode='lines',
                           name='Edges')
    
    # Create node traces
    node_x = [pos[node][0] for node in graph.nodes()]
    node_y = [pos[node][1] for node in graph.nodes()]
    node_communities = [partition[node] for node in graph.nodes()]
    node_degrees = [graph.degree(node) for node in graph.nodes()]
    
    node_trace = go.Scatter(x=node_x, y=node_y,
                           mode='markers',
                           hoverinfo='text',
                           marker=dict(size=8,
                                     color=node_communities,
                                     colorscale='Viridis',
                                     showscale=True,
                                     colorbar=dict(title="Community")),
                           name='Nodes')
    
    # Add hover information
    node_trace.text = [f'Node: {node}<br>Community: {node_communities[i]}<br>Degree: {node_degrees[i]}' 
                       for i, node in enumerate(graph.nodes())]
    
    # Create figure
    fig = go.Figure(data=[edge_trace, node_trace],
                   layout=go.Layout(
                       title=title,
                       titlefont_size=16,
                       showlegend=False,
                       hovermode='closest',
                       margin=dict(b=20,l=5,r=5,t=40),
                       annotations=[ dict(
                           text="Hover over nodes for details. Zoom and pan to explore.",
                           showarrow=False,
                           xref="paper", yref="paper",
                           x=0.005, y=-0.002,
                           xanchor="left", yanchor="bottom",
                           font=dict(size=12)
                       )],
                       xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                       yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                       width=900,
                       height=700))
    
    fig.show()
    return fig

print("Network layout visualization functions defined!")

## Hierarchical Community Structure

In [ ]:
def plot_hierarchy_analysis(louvain_model, title="Hierarchical Community Structure"):
    """Visualize hierarchical community structure across levels."""
    n_levels = len(louvain_model.community_history)
    
    if n_levels == 1:
        print("Only one level found - no hierarchy to display")
        return
    
    fig, axes = plt.subplots(2, min(3, n_levels), figsize=(15, 10))
    if n_levels == 1:
        axes = [axes]
    elif n_levels == 2:
        axes = axes.reshape(-1)
    
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Show first few levels
    levels_to_show = min(3, n_levels)
    
    for level in range(levels_to_show):
        if level < len(louvain_model.level_graphs):
            graph = louvain_model.level_graphs[level]
            partition = louvain_model.community_history[level]
            
            # Get layout
            if len(graph.nodes()) > 100:
                pos = nx.spring_layout(graph, k=1, iterations=20)
            else:
                pos = nx.spring_layout(graph, seed=42)
            
            # Assign colors
            communities = list(set(partition.values()))
            colors = plt.cm.Set3(np.linspace(0, 1, len(communities)))
            node_colors = [colors[partition[node]] for node in graph.nodes()]
            
            # Draw network
            ax_idx = level if n_levels <= 3 else level
            ax = axes[ax_idx] if len(axes.shape) == 1 else axes[0, level]
            
            nx.draw_networkx_nodes(graph, pos, node_color=node_colors, 
                                  node_size=30, alpha=0.8, ax=ax)
            nx.draw_networkx_edges(graph, pos, alpha=0.3, width=0.5, ax=ax)
            
            n_communities = len(communities)
            modularity = louvain_model.modularity_history[level]
            ax.set_title(f'Level {level}\n{n_communities} communities\nQ={modularity:.3f}')
            ax.axis('off')
    
    # Community evolution chart
    if len(axes.shape) == 2 and axes.shape[0] > 1:
        # Plot community count evolution
        community_counts = [len(set(partition.values())) for partition in louvain_model.community_history]
        
        ax_bottom = axes[1, 0] if levels_to_show > 1 else axes[1]
        ax_bottom.plot(range(n_levels), community_counts, marker='o', linewidth=2, markersize=8)
        ax_bottom.set_title('Community Count Evolution')
        ax_bottom.set_xlabel('Hierarchy Level')
        ax_bottom.set_ylabel('Number of Communities')
        ax_bottom.grid(True, alpha=0.3)
        
        # Plot modularity evolution
        if levels_to_show > 2:
            ax_bottom2 = axes[1, 1]
            ax_bottom2.plot(range(n_levels), louvain_model.modularity_history, 
                           marker='s', linewidth=2, markersize=8, color='orange')
            ax_bottom2.set_title('Modularity Evolution')
            ax_bottom2.set_xlabel('Hierarchy Level')
            ax_bottom2.set_ylabel('Modularity')
            ax_bottom2.grid(True, alpha=0.3)
        
        # Community size distribution comparison
        if levels_to_show > 2:
            ax_bottom3 = axes[1, 2]
        else:
            ax_bottom3 = axes[1, 1] if levels_to_show > 1 else axes[1]
        
        # Show size distribution for last level
        last_partition = louvain_model.community_history[-1]
        communities = {}
        for node, comm in last_partition.items():
            if comm not in communities:
                communities[comm] = []
            communities[comm].append(node)
        
        community_sizes = [len(members) for members in communities.values()]
        ax_bottom3.hist(community_sizes, bins=min(10, len(community_sizes)), 
                       alpha=0.7, edgecolor='black')
        ax_bottom3.set_title('Final Community Sizes')
        ax_bottom3.set_xlabel('Community Size')
        ax_bottom3.set_ylabel('Frequency')
        ax_bottom3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def create_hierarchy_sankey_diagram(louvain_model, max_levels=3):
    """Create Sankey diagram showing community evolution across levels."""
    if len(louvain_model.community_history) < 2:
        print("Need at least 2 levels for Sankey diagram")
        return
    
    # This is a simplified version - a full implementation would track
    # how communities split/merge across levels
    
    levels_to_show = min(max_levels, len(louvain_model.community_history))
    
    # Create summary figure instead
    fig, ax = plt.subplots(1, 1, figsize=(12, 6))
    
    levels = range(levels_to_show)
    community_counts = [len(set(louvain_model.community_history[i].values())) 
                       for i in levels]
    modularities = [louvain_model.modularity_history[i] for i in levels]
    
    # Create stacked visualization
    ax2 = ax.twinx()
    
    line1 = ax.plot(levels, community_counts, 'bo-', linewidth=3, markersize=10, 
                   label='Number of Communities')
    line2 = ax2.plot(levels, modularities, 'ro-', linewidth=3, markersize=10, 
                    label='Modularity')
    
    ax.set_xlabel('Hierarchy Level')
    ax.set_ylabel('Number of Communities', color='blue')
    ax2.set_ylabel('Modularity', color='red')
    
    ax.tick_params(axis='y', labelcolor='blue')
    ax2.tick_params(axis='y', labelcolor='red')
    
    ax.set_title('Community Hierarchy Evolution')
    ax.grid(True, alpha=0.3)
    
    # Add legends
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    
    plt.tight_layout()
    plt.show()

print("Hierarchical analysis functions defined!")

## Demonstration on Various Networks

In [ ]:
# Create demonstration networks
print("Creating demonstration networks...")

demo_networks = create_sample_networks()

print(f"Created {len(demo_networks)} demonstration networks:")
for graph, name in demo_networks:
    print(f"  - {name}: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

print("\nNetworks created successfully!")

In [ ]:
# Detailed analysis of Karate Club network
print("Analyzing Karate Club network...")

karate_graph = demo_networks[0][0]

# Apply enhanced Louvain
louvain_karate = EnhancedLouvain(resolution=1.0)
louvain_karate.fit(karate_graph)

print(f"Louvain analysis completed:")
print(f"  Final modularity: {louvain_karate.modularity_:.3f}")
print(f"  Number of hierarchy levels: {louvain_karate.n_levels_}")
print(f"  Number of communities: {len(set(louvain_karate.best_partition_.values()))}")

# Comprehensive visualizations
best_level, best_mod = plot_modularity_evolution(louvain_karate, "Karate Club - Modularity Evolution")
communities, intra, inter = plot_community_analysis(karate_graph, louvain_karate.best_partition_, 
                                                   "Karate Club - Community Analysis")
plot_network_layouts(karate_graph, louvain_karate.best_partition_, 
                    "Karate Club - Network Layouts")
plot_hierarchy_analysis(louvain_karate, "Karate Club - Hierarchical Structure")

In [ ]:
# Interactive visualization of Karate Club
print("Creating interactive visualization for Karate Club...")
fig_karate = create_interactive_network_visualization(karate_graph, louvain_karate.best_partition_,
                                                     'spring', "Interactive Karate Club Network")
print("Interactive Karate Club visualization created!")

In [ ]:
# Compare Louvain performance across different networks
print("Comparing Louvain performance across different networks...")

results_comparison = []

for graph, name in tqdm(demo_networks):
    start_time = time.time()
    
    louvain = EnhancedLouvain(resolution=1.0)
    louvain.fit(graph)
    
    processing_time = time.time() - start_time
    
    n_nodes = graph.number_of_nodes()
    n_edges = graph.number_of_edges()
    n_communities = len(set(louvain.best_partition_.values()))
    modularity = louvain.modularity_
    n_levels = louvain.n_levels_
    
    results_comparison.append({
        'Network': name,
        'Nodes': n_nodes,
        'Edges': n_edges,
        'Communities': n_communities,
        'Modularity': modularity,
        'Levels': n_levels,
        'Time (s)': processing_time
    })

# Create comparison DataFrame
df_comparison = pd.DataFrame(results_comparison)
print("\nLouvain Performance Comparison:")
print(df_comparison.round(3))

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Louvain Performance Comparison Across Networks', fontsize=16, fontweight='bold')

# Modularity comparison
axes[0, 0].bar(df_comparison['Network'], df_comparison['Modularity'], 
               color=plt.cm.viridis(np.linspace(0, 1, len(df_comparison))))
axes[0, 0].set_title('Modularity by Network')
axes[0, 0].set_ylabel('Modularity')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# Number of communities
axes[0, 1].bar(df_comparison['Network'], df_comparison['Communities'], 
               color=plt.cm.plasma(np.linspace(0, 1, len(df_comparison))))
axes[0, 1].set_title('Number of Communities')
axes[0, 1].set_ylabel('Communities')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# Processing time vs network size
axes[1, 0].scatter(df_comparison['Nodes'], df_comparison['Time (s)'], 
                   s=100, alpha=0.7, c=df_comparison['Modularity'], 
                   cmap='coolwarm')
axes[1, 0].set_title('Processing Time vs Network Size')
axes[1, 0].set_xlabel('Number of Nodes')
axes[1, 0].set_ylabel('Time (seconds)')
axes[1, 0].grid(True, alpha=0.3)

# Hierarchy levels
axes[1, 1].bar(df_comparison['Network'], df_comparison['Levels'], 
               color=plt.cm.Set3(np.linspace(0, 1, len(df_comparison))))
axes[1, 1].set_title('Hierarchy Levels')
axes[1, 1].set_ylabel('Number of Levels')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Network from Data Points Demonstration

In [ ]:
# Create network from clustered data points
print("Creating network from data points...")

# Generate sample data with clear clusters
X_data, y_true = make_blobs(n_samples=200, centers=4, cluster_std=1.5, 
                           center_box=(-10, 10), random_state=42)

# Create network using different methods
methods = ['knn', 'threshold']
method_results = []

for method in methods:
    if method == 'knn':
        graph_data = create_network_from_data(X_data, method='knn', k=8)
    else:
        graph_data = create_network_from_data(X_data, method='threshold', threshold=3.0)
    
    # Apply Louvain
    louvain_data = EnhancedLouvain(resolution=1.0)
    louvain_data.fit(graph_data)
    
    # Calculate ARI with true labels
    ari = adjusted_rand_score(y_true, louvain_data.labels_)
    nmi = normalized_mutual_info_score(y_true, louvain_data.labels_)
    
    method_results.append({
        'method': method,
        'graph': graph_data,
        'louvain': louvain_data,
        'ari': ari,
        'nmi': nmi
    })
    
    print(f"{method.upper()} method:")
    print(f"  Nodes: {graph_data.number_of_nodes()}, Edges: {graph_data.number_of_edges()}")
    print(f"  Communities: {len(set(louvain_data.best_partition_.values()))}")
    print(f"  Modularity: {louvain_data.modularity_:.3f}")
    print(f"  ARI: {ari:.3f}, NMI: {nmi:.3f}")
    print()

# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Louvain on Networks from Data Points', fontsize=16, fontweight='bold')

# Original data
scatter = axes[0, 0].scatter(X_data[:, 0], X_data[:, 1], c=y_true, 
                            cmap='tab10', alpha=0.7, s=50)
axes[0, 0].set_title('True Clusters')
axes[0, 0].set_xlabel('Feature 1')
axes[0, 0].set_ylabel('Feature 2')

# Results for each method
for i, result in enumerate(method_results):
    # Community detection result
    axes[0, i+1].scatter(X_data[:, 0], X_data[:, 1], c=result['louvain'].labels_, 
                        cmap='tab10', alpha=0.7, s=50)
    axes[0, i+1].set_title(f'{result["method"].upper()} Communities\nARI: {result["ari"]:.3f}')
    axes[0, i+1].set_xlabel('Feature 1')
    axes[0, i+1].set_ylabel('Feature 2')
    
    # Network visualization
    graph = result['graph']
    pos = nx.get_node_attributes(graph, 'pos')
    
    # Draw network with community colors
    partition = result['louvain'].best_partition_
    communities = list(set(partition.values()))
    colors = plt.cm.Set3(np.linspace(0, 1, len(communities)))
    node_colors = [colors[partition[node]] for node in graph.nodes()]
    
    nx.draw_networkx_nodes(graph, pos, node_color=node_colors, 
                          node_size=30, alpha=0.8, ax=axes[1, i+1])
    nx.draw_networkx_edges(graph, pos, alpha=0.2, width=0.5, ax=axes[1, i+1])
    
    axes[1, i+1].set_title(f'{result["method"].upper()} Network\nQ: {result["louvain"].modularity_:.3f}')
    axes[1, i+1].axis('off')

# Performance comparison
methods_names = [r['method'].upper() for r in method_results]
ari_scores = [r['ari'] for r in method_results]
nmi_scores = [r['nmi'] for r in method_results]
modularities = [r['louvain'].modularity_ for r in method_results]

x = np.arange(len(methods_names))
width = 0.25

axes[1, 0].bar(x - width, ari_scores, width, label='ARI', alpha=0.8)
axes[1, 0].bar(x, nmi_scores, width, label='NMI', alpha=0.8)
axes[1, 0].bar(x + width, modularities, width, label='Modularity', alpha=0.8)

axes[1, 0].set_title('Performance Comparison')
axes[1, 0].set_xlabel('Method')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(methods_names)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best method by ARI: {method_results[np.argmax(ari_scores)]['method'].upper()}")
print(f"Best method by modularity: {method_results[np.argmax(modularities)]['method'].upper()}")

## Summary and Conclusions

This notebook demonstrates advanced Louvain method implementation with comprehensive network visualizations including:

1. **Network Layout Graphs**: Multiple force-directed and structured layouts showing community structure
2. **Modularity Evolution**: Track modularity optimization over hierarchical levels
3. **Community Size Distribution**: Analysis of detected community structures
4. **Hierarchical Community Structure**: Multi-level community detection visualization
5. **Interactive Network Graphs**: Plotly-based interactive visualizations with zoom, pan, and hover
6. **Performance Comparisons**: Analysis across different network types and construction methods

### Key Insights:
- **Hierarchical Structure**: Louvain naturally discovers multi-level community organization
- **Modularity Optimization**: The algorithm iteratively improves modularity until convergence
- **Network Construction**: The method of creating networks from data significantly impacts results
- **Scalability**: Efficient performance on networks with hundreds to thousands of nodes
- **Quality Metrics**: ARI and NMI provide validation against known ground truth

### Applications:
- **Social Network Analysis**: Finding communities in social media, collaboration networks
- **Biological Networks**: Protein interaction modules, gene regulatory networks
- **Web Analytics**: Website clustering, user behavior analysis
- **Transportation**: Route optimization, traffic flow analysis
- **Recommendation Systems**: User and item clustering for collaborative filtering

### Best Practices:
- **Resolution Parameter**: Tune resolution to control community size granularity
- **Network Construction**: Choose appropriate method (KNN vs threshold) based on data characteristics
- **Validation**: Use multiple metrics (modularity, ARI, NMI) for comprehensive evaluation
- **Visualization**: Interactive plots enable better exploration of large networks
- **Hierarchy**: Analyze multiple levels to understand community organization at different scales